# Vietnamese Retrieval Model Sweep on Kaggle

This notebook runs model inference only. It does **not** connect to Qdrant.

It creates and stores:

- BGE-M3 sparse vectors for 297 context chunks and 100 queries.
- BGE-M3 dense vectors for 297 context chunks and 100 queries.
- Qwen3-Embedding-0.6B dense vectors for the same documents and queries.
- Qwen3-Embedding-4B dense vectors for the same documents and queries.
- Scores from three rerankers over all `100 x 297 = 29,700` query-chunk pairs:
  - `BAAI/bge-reranker-v2-m3`
  - `Qwen/Qwen3-Reranker-0.6B`
  - `Qwen/Qwen3-Reranker-4B`

Models are loaded and released sequentially to fit a Kaggle GPU. Every model logs load time, inference time, item count, throughput, and total measured time.

Attach the prepared `ViRHE4QA_v2_all_contexts_297` folder as a Kaggle Dataset before running the notebook.


In [ ]:
# Kaggle setup. The first run downloads Python packages and model weights.
!pip install -q -U 'FlagEmbedding>=1.3,<2' 'sentence-transformers>=2.7' accelerate pandas pyarrow tqdm

In [ ]:
from pathlib import Path
import gc
import glob
import hashlib
import json
import os
import time
import unicodedata

import numpy as np
import pandas as pd
import torch

SEED = 42
BGE_MODEL_NAME = 'BAAI/bge-m3'
QWEN_EMBEDDING_MODELS = {
    'qwen3_embedding_0_6b': {
        'model_name': 'Qwen/Qwen3-Embedding-0.6B',
        'expected_dim': 1024,
        'batch_size': 8,
    },
    'qwen3_embedding_4b': {
        'model_name': 'Qwen/Qwen3-Embedding-4B',
        'expected_dim': 2560,
        'batch_size': 1,
    },
}
RERANKER_MODELS = {
    'bge_reranker_v2_m3': {
        'model_name': 'BAAI/bge-reranker-v2-m3',
        'batch_size': 32,
        'prompt_name': None,
    },
    'qwen3_reranker_0_6b': {
        'model_name': 'Qwen/Qwen3-Reranker-0.6B',
        'batch_size': 16,
        'prompt_name': 'query',
    },
    'qwen3_reranker_4b': {
        'model_name': 'Qwen/Qwen3-Reranker-4B',
        'batch_size': 4,
        'prompt_name': 'query',
    },
}
BGE_BATCH_SIZE = 8
MAX_LENGTH_BGE = 8192
MAX_LENGTH_EMBEDDING = 8192
MAX_LENGTH_RERANKER = 2048
OUTPUT_DIR = Path('/kaggle/working/virhe4qa_model_sweep')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

timing_records = []

def timed_call(model_name, phase, fn, item_count=None):
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    started = time.perf_counter()
    value = fn()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - started
    record = {
        'model': model_name,
        'phase': phase,
        'seconds': round(elapsed, 4),
        'items': item_count,
        'items_per_second': round(item_count / elapsed, 4) if item_count and elapsed else None,
    }
    timing_records.append(record)
    suffix = '' if not item_count else f" | {item_count} items | {record['items_per_second']:.2f}/s"
    print(f"[timing] {model_name:<30} {phase:<22} {elapsed:>10.3f}s{suffix}")
    return value

def release_model(model):
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# Locate the attached Kaggle Dataset.
chunk_candidates = []
query_candidates = []
for root in ('/kaggle/input', '/kaggle/working'):
    if os.path.exists(root):
        chunk_candidates.extend(glob.glob(root + '/**/ViRHE4QA_v2_all_contexts.csv', recursive=True))
        chunk_candidates.extend(glob.glob(root + '/**/ViRHE4QA_v2.csv', recursive=True))
        query_candidates.extend(glob.glob(root + '/**/queries_100.csv', recursive=True))
        query_candidates.extend(glob.glob(root + '/**/ViRHE4QA_v2_sample_100.csv', recursive=True))

chunk_candidates = sorted(set(chunk_candidates))
query_candidates = sorted(set(query_candidates))
if not chunk_candidates:
    raise FileNotFoundError('Attach the prepared chunks Kaggle Dataset first.')
if not query_candidates:
    raise FileNotFoundError('Attach the 100-question CSV Kaggle Dataset first.')

preferred_chunks = [p for p in chunk_candidates if p.endswith('ViRHE4QA_v2_all_contexts.csv')]
preferred_queries = [p for p in query_candidates if p.endswith('queries_100.csv')]
CHUNK_CSV = Path((preferred_chunks or chunk_candidates)[0])
QUERY_CSV = Path((preferred_queries or query_candidates)[0])
print('Chunks:', CHUNK_CSV)
print('Queries:', QUERY_CSV)


In [ ]:
# Build deterministic manifests: 297 context chunks and 100 query questions.
def normalize_context(value: str) -> str:
    value = (value or '').replace('\r\n', '\n').replace('\r', '\n').strip()
    return unicodedata.normalize('NFC', value)

def stable_document_id(text: str) -> str:
    digest = hashlib.sha256(text.encode('utf-8')).hexdigest()[:20]
    return f'virhe4qa_{digest}'

chunk_source_df = pd.read_csv(CHUNK_CSV, dtype=str, keep_default_na=False)
chunk_source_df['context_norm'] = chunk_source_df['context'].map(normalize_context)
chunk_source_df = chunk_source_df[chunk_source_df['context_norm'].ne('')].copy()
chunk_source_df['_index_num'] = pd.to_numeric(chunk_source_df.get('index', ''), errors='coerce').fillna(-1)
chunk_source_df = chunk_source_df.sort_values(['context_norm', '_index_num', 'split', 'question'])
chunks = chunk_source_df.drop_duplicates('context_norm', keep='first').sort_values('context_norm').reset_index(drop=True)
chunks['chunk_id'] = [f'chunk_{i:04d}' for i in range(1, len(chunks) + 1)]
chunks['document_id'] = chunks['context_norm'].map(stable_document_id)
chunks['text'] = chunks['context_norm']
chunks['point_key'] = chunks['document_id'] + ':' + chunks['chunk_id']
manifest = chunks[['chunk_id', 'document_id', 'point_key', 'split', 'index', 'label', 'text']].copy()
manifest.to_csv(OUTPUT_DIR / 'chunks_manifest.csv', index=False, encoding='utf-8')

query_source_df = pd.read_csv(QUERY_CSV, dtype=str, keep_default_na=False)
query_source_df['context_norm'] = query_source_df['context'].map(normalize_context)
query_source_df['question_norm'] = query_source_df['question'].map(lambda value: (value or '').strip())
query_source_df = query_source_df[query_source_df['question_norm'].ne('')].copy()
query_source_df = query_source_df.drop_duplicates('question_norm').sort_values(['question_norm', 'context_norm']).head(100).copy()
context_to_document = dict(zip(chunks['context_norm'], chunks['document_id']))
context_to_chunk = dict(zip(chunks['context_norm'], chunks['chunk_id']))
query_source_df['query_id'] = [f'query_{i:03d}' for i in range(1, len(query_source_df) + 1)]
query_source_df['gold_document_id'] = query_source_df['context_norm'].map(context_to_document)
query_source_df['gold_chunk_id'] = query_source_df['context_norm'].map(context_to_chunk)
if query_source_df['gold_document_id'].isna().any():
    raise ValueError('Every query context must exist in the 297-chunk manifest')
query_manifest = query_source_df[['query_id', 'gold_document_id', 'gold_chunk_id', 'split', 'index', 'label', 'question_norm', 'context_norm']].rename(columns={'question_norm': 'question', 'context_norm': 'gold_context'})
query_manifest.to_csv(OUTPUT_DIR / 'queries_manifest.csv', index=False, encoding='utf-8')

# Precompute the exact 100 x 297 reranker pair order used by every reranker.
pair_manifest = pd.DataFrame({
    'query_id': np.repeat(query_manifest['query_id'].to_numpy(), len(manifest)),
    'chunk_id': np.tile(manifest['chunk_id'].to_numpy(), len(query_manifest)),
})
gold_by_query = dict(zip(query_manifest['query_id'], query_manifest['gold_chunk_id']))
pair_manifest['is_gold'] = [chunk_id == gold_by_query[query_id] for query_id, chunk_id in zip(pair_manifest['query_id'], pair_manifest['chunk_id'])]
pair_manifest.to_csv(OUTPUT_DIR / 'reranker_pair_manifest.csv', index=False, encoding='utf-8')

print('unique context chunks:', len(manifest))
print('query questions:', len(query_manifest))
print('reranker pairs:', len(pair_manifest))
assert len(manifest) == 297
assert len(query_manifest) == 100
assert len(pair_manifest) == 29700
display(manifest.head(2))
display(query_manifest.head(2))
display(pair_manifest.head(2))


## 1. BGE-M3 sparse + dense

BGE-M3 is the only sparse encoder in this sweep. Its dense output is 1,024-dimensional; its sparse output is stored as variable-length token IDs and lexical weights.

In [ ]:
from FlagEmbedding import BGEM3FlagModel

bge_model = timed_call(
    'BAAI/bge-m3',
    'load model',
    lambda: BGEM3FlagModel(BGE_MODEL_NAME, use_fp16=torch.cuda.is_available()),
)

def bge_encode(texts, phase, item_count):
    return timed_call(
        'BAAI/bge-m3',
        phase,
        lambda: bge_model.encode(
            texts,
            batch_size=BGE_BATCH_SIZE,
            max_length=MAX_LENGTH_BGE,
            return_dense=True,
            return_sparse=True,
            return_colbert_vecs=False,
        ),
        item_count=item_count,
    )

def sparse_rows(lexical_weights):
    rows = []
    for weights in lexical_weights:
        pairs = sorted((int(token_id), float(weight)) for token_id, weight in weights.items() if float(weight) != 0.0)
        rows.append({'indices': [token_id for token_id, _ in pairs], 'values': [weight for _, weight in pairs]})
    return rows

bge_document_outputs = bge_encode(manifest['text'].tolist(), 'encode documents', len(manifest))
bge_query_outputs = bge_encode(query_manifest['question'].tolist(), 'encode queries', len(query_manifest))
bge_dense = np.asarray(bge_document_outputs['dense_vecs'], dtype=np.float32)
bge_query_dense = np.asarray(bge_query_outputs['dense_vecs'], dtype=np.float32)
bge_sparse = sparse_rows(bge_document_outputs['lexical_weights'])
bge_query_sparse = sparse_rows(bge_query_outputs['lexical_weights'])
print('BGE document dense:', bge_dense.shape)
print('BGE query dense:', bge_query_dense.shape)
print('BGE document sparse rows:', len(bge_sparse))
print('BGE query sparse rows:', len(bge_query_sparse))


In [ ]:
# Save BGE-M3 vectors.
def write_sparse_jsonl(path, ids, sparse_rows_value):
    with path.open('w', encoding='utf-8') as stream:
        for item_id, sparse in zip(ids, sparse_rows_value):
            stream.write(json.dumps({'id': item_id, 'indices': sparse['indices'], 'values': sparse['values']}, ensure_ascii=False) + '\n')

np.save(OUTPUT_DIR / 'bge_m3_dense.npy', bge_dense)
np.save(OUTPUT_DIR / 'bge_m3_query_dense.npy', bge_query_dense)
write_sparse_jsonl(OUTPUT_DIR / 'bge_m3_sparse.jsonl', manifest['chunk_id'].tolist(), bge_sparse)
write_sparse_jsonl(OUTPUT_DIR / 'bge_m3_query_sparse.jsonl', query_manifest['query_id'].tolist(), bge_query_sparse)
(OUTPUT_DIR / 'bge_m3_metadata.json').write_text(json.dumps({
    'model': BGE_MODEL_NAME,
    'num_documents': len(manifest),
    'num_queries': len(query_manifest),
    'dense_dimension': int(bge_dense.shape[1]),
    'sparse_format': 'jsonl indices + values; variable nnz per item',
}, indent=2), encoding='utf-8')


In [ ]:
release_model(bge_model)
print('BGE-M3 released')

## 2. Dense embeddings: Qwen3-Embedding-0.6B and 4B

Both Qwen embedding models produce dense vectors only. Documents are encoded without a prompt; queries use the model's built-in `query` prompt.

In [ ]:
from sentence_transformers import SentenceTransformer

def run_qwen_embedding(alias, spec):
    model_name = spec['model_name']
    model = timed_call(
        model_name,
        'load model',
        lambda: SentenceTransformer(
            model_name,
            device='cuda' if torch.cuda.is_available() else 'cpu',
            model_kwargs={'torch_dtype': torch.float16} if torch.cuda.is_available() else {},
            tokenizer_kwargs={'padding_side': 'left'},
        ),
    )
    model.max_seq_length = MAX_LENGTH_EMBEDDING
    encode_kwargs = {
        'batch_size': spec['batch_size'],
        'normalize_embeddings': True,
        'show_progress_bar': True,
    }
    document_vectors = timed_call(
        model_name,
        'encode documents',
        lambda: model.encode(manifest['text'].tolist(), **encode_kwargs),
        item_count=len(manifest),
    )
    query_vectors = timed_call(
        model_name,
        'encode queries',
        lambda: model.encode(query_manifest['question'].tolist(), prompt_name='query', **encode_kwargs),
        item_count=len(query_manifest),
    )
    document_vectors = np.asarray(document_vectors, dtype=np.float32)
    query_vectors = np.asarray(query_vectors, dtype=np.float32)
    if document_vectors.shape[1] != spec['expected_dim']:
        raise ValueError(f'{model_name}: expected {spec["expected_dim"]} dimensions, got {document_vectors.shape[1]}')
    np.save(OUTPUT_DIR / f'{alias}_dense.npy', document_vectors)
    np.save(OUTPUT_DIR / f'{alias}_query_dense.npy', query_vectors)
    (OUTPUT_DIR / f'{alias}_metadata.json').write_text(json.dumps({
        'model': model_name,
        'num_documents': len(manifest),
        'num_queries': len(query_manifest),
        'dense_dimension': int(document_vectors.shape[1]),
        'normalized': True,
        'query_prompt_name': 'query',
    }, indent=2), encoding='utf-8')
    print(f'{model_name} document dense:', document_vectors.shape)
    print(f'{model_name} query dense:', query_vectors.shape)
    release_model(model)
    return document_vectors, query_vectors

qwen06_dense, qwen06_query_dense = run_qwen_embedding('qwen3_embedding_0_6b', QWEN_EMBEDDING_MODELS['qwen3_embedding_0_6b'])


In [ ]:
qwen4_dense, qwen4_query_dense = run_qwen_embedding('qwen3_embedding_4b', QWEN_EMBEDDING_MODELS['qwen3_embedding_4b'])

## 3. Reranker scores for all 29,700 pairs

Each reranker receives the same pair order: query-major, then chunk order. Scores are normalized to `[0, 1]` with a sigmoid so the three model outputs are easy to compare. Raw model-specific scores are not discarded from the timing/metadata, but the saved score files contain the normalized values.

In [ ]:
from sentence_transformers import CrossEncoder
from tqdm.auto import tqdm

query_text_by_id = dict(zip(query_manifest['query_id'], query_manifest['question']))
chunk_text_by_id = dict(zip(manifest['chunk_id'], manifest['text']))
pair_tuples = [
    (query_text_by_id[query_id], chunk_text_by_id[chunk_id])
    for query_id, chunk_id in zip(pair_manifest['query_id'], pair_manifest['chunk_id'])
]


def score_pairs(model, pairs, batch_size, prompt_name=None):
    score_batches = []
    for start in tqdm(range(0, len(pairs), batch_size), desc='reranking pairs'):
        batch = pairs[start:start + batch_size]
        kwargs = {
            'batch_size': batch_size,
            'show_progress_bar': False,
            'activation_fn': torch.nn.Sigmoid(),
        }
        if prompt_name:
            kwargs['prompt_name'] = prompt_name
        scores = model.predict(batch, **kwargs)
        score_batches.append(np.asarray(scores, dtype=np.float32).reshape(-1))
    return np.concatenate(score_batches)

for alias, spec in RERANKER_MODELS.items():
    model_name = spec['model_name']
    load_kwargs = {
        'torch_dtype': torch.float16,
    } if torch.cuda.is_available() else {}
    reranker = timed_call(
        model_name,
        'load model',
        lambda model_name=model_name, load_kwargs=load_kwargs: CrossEncoder(
            model_name,
            device='cuda' if torch.cuda.is_available() else 'cpu',
            max_length=MAX_LENGTH_RERANKER,
            model_kwargs=load_kwargs,
        ),
    )
    scores = timed_call(
        model_name,
        'score 29,700 pairs',
        lambda reranker=reranker, spec=spec: score_pairs(
            reranker,
            pair_tuples,
            batch_size=spec['batch_size'],
            prompt_name=spec['prompt_name'],
        ),
        item_count=len(pair_tuples),
    )
    score_matrix = scores.reshape(len(query_manifest), len(manifest))
    np.save(OUTPUT_DIR / f'{alias}_scores.npy', score_matrix)
    pair_scores = pair_manifest.copy()
    pair_scores['score'] = scores
    pair_scores.to_csv(OUTPUT_DIR / f'{alias}_scores.csv', index=False, encoding='utf-8')
    (OUTPUT_DIR / f'{alias}_metadata.json').write_text(json.dumps({
        'model': model_name,
        'num_queries': len(query_manifest),
        'num_chunks': len(manifest),
        'num_pairs': len(pair_tuples),
        'score_shape': list(score_matrix.shape),
        'score_range': '[0, 1] sigmoid-normalized',
        'pair_order': 'query-major then chunk order',
        'prompt_name': spec['prompt_name'],
    }, indent=2), encoding='utf-8')
    print(f'{model_name} scores:', score_matrix.shape, 'range=', (float(scores.min()), float(scores.max())))
    release_model(reranker)


In [ ]:
# Timing summary, integrity checks, and archive.
model_names = [BGE_MODEL_NAME] + [spec['model_name'] for spec in QWEN_EMBEDDING_MODELS.values()] + [spec['model_name'] for spec in RERANKER_MODELS.values()]
for model_name in model_names:
    total_seconds = sum(row['seconds'] for row in timing_records if row['model'] == model_name)
    timing_records.append({
        'model': model_name,
        'phase': 'total measured time',
        'seconds': round(total_seconds, 4),
        'items': None,
        'items_per_second': None,
    })
    print(f'[timing] {model_name:<30} {"total measured time":<22} {total_seconds:>10.3f}s')

timing_df = pd.DataFrame(timing_records)
timing_df.to_csv(OUTPUT_DIR / 'model_timings.csv', index=False)
(OUTPUT_DIR / 'model_timings.json').write_text(json.dumps(timing_records, indent=2), encoding='utf-8')
display(timing_df)

assert len(manifest) == 297
assert len(query_manifest) == 100
assert len(pair_manifest) == 29700
assert bge_dense.shape == (297, 1024)
assert bge_query_dense.shape == (100, 1024)
assert qwen06_dense.shape == (297, 1024)
assert qwen06_query_dense.shape == (100, 1024)
assert qwen4_dense.shape == (297, 2560)
assert qwen4_query_dense.shape == (100, 2560)
assert len(bge_sparse) == 297
assert len(bge_query_sparse) == 100

for alias in RERANKER_MODELS:
    score_matrix = np.load(OUTPUT_DIR / f'{alias}_scores.npy')
    assert score_matrix.shape == (100, 297), (alias, score_matrix.shape)
    assert np.isfinite(score_matrix).all(), alias

print('All embedding and reranker artifacts passed shape checks.')
for path_item in sorted(OUTPUT_DIR.iterdir()):
    print(f'  {path_item.name}: {path_item.stat().st_size / 1024 / 1024:.2f} MB')

import shutil
archive_path = shutil.make_archive('/kaggle/working/virhe4qa_model_sweep', 'zip', OUTPUT_DIR)
print('Download:', archive_path)


## Output mapping for later Qdrant and retrieval evaluation

Embedding outputs:

- `bge_m3_dense.npy` / `bge_m3_query_dense.npy`: BGE-M3 dense vectors, dimension 1,024.
- `bge_m3_sparse.jsonl` / `bge_m3_query_sparse.jsonl`: BGE-M3 sparse token IDs and lexical weights.
- `qwen3_embedding_0_6b_dense.npy` / `qwen3_embedding_0_6b_query_dense.npy`: Qwen 0.6B dense vectors, dimension 1,024.
- `qwen3_embedding_4b_dense.npy` / `qwen3_embedding_4b_query_dense.npy`: Qwen 4B dense vectors, dimension 2,560.

Reranker outputs:

- Each `*_scores.npy` has shape `(100, 297)`, indexed by `queries_manifest.csv` row and `chunks_manifest.csv` row.
- Each `*_scores.csv` contains `query_id`, `chunk_id`, `is_gold`, and normalized `score` for all 29,700 pairs.
- `*_metadata.json` records model name, shape, pair order, and score normalization.

All dense vector families should be stored as separate named vectors or separate Qdrant collections because their dimensions differ. Reranker scores are not vectors and should remain an offline evaluation artifact or be used only after candidate retrieval.